# Async/Await Fundamentals

**Topic**: Asynchronous Programming with async/await  
**Goal**: Master concurrent programming for I/O-bound operations

## What You'll Learn
- What asynchronous programming is
- Difference between concurrency and parallelism
- When to use async (I/O-bound vs CPU-bound)
- The event loop and how it works
- Basic async/await syntax
- Running multiple tasks concurrently
- Async context managers and iterators
- Error handling in async code
- Common pitfalls and how to avoid them

## Setup

We'll use Python's built-in `asyncio` library for asynchronous programming.

In [5]:
%pip install aiohttp

  Using cached attrs-25.4.0-py3-none-any.whl.metadata (10 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.1/75.1 kB 2.2 MB/s eta 0:00:00
  Using cached idna-3.11-py3-none-any.whl.metadata (8.4 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 493.2/493.2 kB 9.5 MB/s eta 0:00:0010.6 MB/s eta 0:00:01
Using cached attrs-25.4.0-py3-none-any.whl (67 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 123.2 kB/s eta 0:00:001m57.3 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 kB 1.0 MB/s eta 0:00:00? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.0/95.0 kB 2.8 MB/s eta 0:00:00
Using cached idna-3.11-py3-none-any.whl (71 kB)

[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [6]:
import asyncio
import time
from typing import List
import aiohttp

print("asyncio imported successfully")
print(f"Python version: {asyncio.__version__ if hasattr(asyncio, '__version__') else 'built-in'}")



asyncio imported successfully
Python version: built-in


## 1. Understanding Synchronous vs Asynchronous

Let's compare synchronous (blocking) vs asynchronous (non-blocking) execution.

### Synchronous Execution
Each operation blocks until complete. Total time = sum of all operations.


In [22]:
#synchronous example - operations run one after another
def sync_task(name, delay):
    """Simulate a blocking operation"""
    print(f"[Sync] {name}: Starting (will take {delay}s)")
    time.sleep(delay)   # Blocks the entire program
    print(f"[Sync] {name}: Done")
    return f"{name} completed"

# Run synchronously
start = time.time()

result1 = sync_task("Task 1", 2)
result2 = sync_task("Task 2", 2)
result3 = sync_task("Task 3", 2)

elapsed = time.time() - start
print(f"\nTotal time (synchronously): {elapsed:.2f}s")

[Sync] Task 1: Starting (will take 2s)
[Sync] Task 1: Done
[Sync] Task 2: Starting (will take 2s)
[Sync] Task 2: Done
[Sync] Task 3: Starting (will take 2s)
[Sync] Task 3: Done

Total time (synchronously): 6.01s


### Asynchronous Execution
Tasks can run concurrently. Total time ≈ longest task (not sum).


In [21]:
# Asynchronous example - operations run concurrently
async def async_task(name, delay):
    """Simulate a non-blocking operation"""
    print(f"[Async] {name}: starting (will take {delay}s)")
    await asyncio.sleep(delay)  # Non-blocking
    print(f"[Async] {name}: Done")
    return f"{name} completed"

async def run_async_tasks():
    """Run multiple tasks concurrently"""
    start = time.time()

    # Run all tasks concurrently
    results = await asyncio.gather(
        async_task("Task 1", 1),
        async_task("Task 2", 3),
        async_task("Task 3", 2)
    )

    elapsed = time.time() - start
    print(f"\nTotal time (asynchronous): {elapsed:.2f}s")
    return results

# Run the async function
await run_async_tasks()

[Async] Task 1: starting (will take 1s)
[Async] Task 2: starting (will take 3s)
[Async] Task 3: starting (will take 2s)
[Async] Task 1: Done
[Async] Task 3: Done
[Async] Task 2: Done

Total time (asynchronous): 3.00s


['Task 1 completed', 'Task 2 completed', 'Task 3 completed']

**Key Observation:**
- Synchronous: 6 seconds (tasks run one after another)
- Asynchronous: 3 seconds (tasks run concurrently)

This is the power of async for I/O-bound operations!

## 2. Concurrency vs Parallelism

Understanding this distinction is critical.

### Concurrency (Async)
- **One chef** switching between multiple dishes
- Tasks make progress together (but one at a time)
- Perfect for I/O-bound work (waiting for network, disk, etc.)

### Parallelism (Threads/Processes)
- **Multiple chefs** working simultaneously
- Tasks run truly at the same time
- Perfect for CPU-bound work (computation, processing)

### Concurrency Demo

In [17]:
# Demonstrate concurrency (async)
async def demonstrate_concurrency():
    """Show how async handles multiple tasks on a single thread"""

    async def task(name, steps, delay=0.5):
        for i in range(steps):
            print(f"{name}: Step {i+1}")
            await asyncio.sleep(delay)    # Yield control to other tasks

    print("Concurrency Demo")
    print("Watch how tasks interleave (switching between them):\n")

    await asyncio.gather(
        task("Task A", 3, 0.9),
        task("Task B", 3, 0.3),
        task("Task C", 4, 0.2),
    )

    print("\nAll tasks ran concurrently on a single thread")

await demonstrate_concurrency()

Concurrency Demo
Watch how tasks interleave (switching between them):

Task A: Step 1
Task B: Step 1
Task C: Step 1
Task C: Step 2
Task B: Step 2
Task C: Step 3
Task B: Step 3
Task C: Step 4
Task A: Step 2
Task A: Step 3

All tasks ran concurrently on a single thread


**What Happened:**
- Tasks A, B, and C interleave (you'll see them mixed together)
- Only one task executes at any moment
- When one task hits `await`, another task runs
- This is **concurrency**: dealing with many things at once

## 3. I/O-Bound vs CPU-Bound Tasks

Knowing when to use async is critical.

### I/O-Bound (Use Async ✓)
Tasks spend most time **waiting** for external operations:
- Network requests (API calls)
- Database queries
- File operations
- User input

### CPU-Bound (Don't Use Async ✗)
Tasks spend most time **computing**:
- Mathematical calculations
- Image/video processing
- Data encryption
- Machine learning

### I/O-Bound Example

In [20]:
# I/O-Bound Example - Async helps here
async def io_bound_example():
    """Simulate I/O-bound operations (network requests)"""

    async def fetch_data(source, delay):
        print(f"Fetching from {source}")
        await asyncio.sleep(delay)  # Simulating network delay
        print(f"Got data from {source}")
        return f"Data from {source}"

    start = time.time()

    # Fetch from 5 sources concurrently
    results = await asyncio.gather(
        fetch_data("API-1", 1.0),
        fetch_data("API-2", 1.5),
        fetch_data("API-3", 0.8),
        fetch_data("API-4", 1.2),
        fetch_data("API-5", 1.0)
    )

    elapsed = time.time() - start
    print(f"\nFetched {len(results)} sources in {elapsed:.2f}s")
    print(f"If done sequentially: ~5.5s")
    print(f"With async: ~{elapsed:.2f}s (max of individual delays)")

await io_bound_example()

Fetching from API-1
Fetching from API-2
Fetching from API-3
Fetching from API-4
Fetching from API-5
Got data from API-3
Got data from API-1
Got data from API-5
Got data from API-4
Got data from API-2

Fetched 5 sources in 1.50s
If done sequentially: ~5.5s
With async: ~1.50s (max of individual delays)


### CPU-Bound Example

In [28]:
# CPU-Bound Example - Async doesn't help here
def cpu_bound_sync(n):
    """CPU-intensive calculation (synchronous)"""
    total = 0
    for i in range(n):
        total += i ** 2
    return total

async def cpu_bound_async(n):
    """same calculation, but marked as async (doesn't help)"""
    total = 0
    for i in range(n):
        total += i ** 2
    return total

# compare sync vs async for CPU-bound work
n = 10000000

print("CPU-Bound task")

# Synchronous
start = time.time()
result_sync = cpu_bound_sync(n)
time_sync = time.time() - start
print(f"Synchronous: {time_sync:.3f}s")

# Asynchronous (won't be faster)
start = time.time()
result_async = await cpu_bound_async(n)
time_async = time.time() - start
print(f"Asynchronous: {time_async:.3f}s")

print(f"\nDifference: {abs(time_async - time_sync):.3f}s")
print("Async provides NO benefit for CPU-bound tasks")

CPU-Bound task
Synchronous: 0.382s
Asynchronous: 0.377s

Difference: 0.005s
Async provides NO benefit for CPU-bound tasks


> Run the above cell multiple time - there will different delay multiple time

## 4. The Event Loop

The event loop is the heart of async programming. It manages and schedules all async operations.

**How it works:**
1. Keeps track of all pending tasks
2. Runs tasks one at a time
3. When a task hits `await`, switches to another task
4. Continues until all tasks complete



### Understanding the event loop

In [29]:
# Understanding the event loop
async def understand_event_loop():
    """Demonstrate how the event loop switches between  tasks"""

    async def task_with_logging(name, delay):
        print(f"[{time.strftime('%H:%M:%S')}] {name}: Started")
        await asyncio.sleep(delay)
        print(f"[{time.strftime('%H:%M:%S')}] {name}: Finished")
        return f"{name} done"

    print("Event Loop Execution")
    print("Watch the timestamps to see task interleaving:\n")

    # create tasks
    task1 = asyncio.create_task(task_with_logging("Task 1", 2.0))
    task2 = asyncio.create_task(task_with_logging("Task 2", 1.0))
    task3 = asyncio.create_task(task_with_logging("Task 3", 1.5))

    # Wait for all to complete
    results = await asyncio.gather(task1, task2, task3)

    print(f"\nResults: {results}")

await understand_event_loop()

Event Loop Execution
Watch the timestamps to see task interleaving:

[12:39:35] Task 1: Started
[12:39:35] Task 2: Started
[12:39:35] Task 3: Started
[12:39:36] Task 2: Finished
[12:39:37] Task 3: Finished
[12:39:37] Task 1: Finished

Results: ['Task 1 done', 'Task 2 done', 'Task 3 done']


**Observations:**
- All tasks start at roughly the same time
- Task 2 (1.0s) finishes first
- Task 3 (1.5s) finishes second
- Task 1 (2.0s) finishes last
- Total time ≈ 2 seconds (longest task), not 4.5 seconds (sum)

## 5. Basic Async/Await Syntax

### Key Concepts:
- `async def` creates a coroutine function
- Calling a coroutine returns a coroutine object (doesn't execute yet)
- `await` pauses the coroutine and waits for result
- Always `await` your coroutines!

In [ ]:
async def greet(name):
    """Simple async function"""
    await asyncio.sleep(1)    # Simulate some async work
    return f"Hello {name}"

async def greet_multiple():
    """Demonstrate basic async/await usage"""

    # Sequential execution (one after another)
    print("Sequential Execution")
    start = time.time()

    greeting1 = await greet("Alice")
    print(greeting1)

    greeting2 = await greet("Bob")
    print(greeting2)

    greeting3 = await greet("Charlie")
    print(greeting3)

    elapsed = time.time() - start
    print(f"Time: {elapsed:.2f}s (each greeting waits for previous)\n")

    # Concurrent execution (all at once)
    print("Concurrent Execution")
    start = time.time()

    results = await asyncio.gather(
        greet("Alice"),
        greet("Bob"),
        greet("Charlie")
    )

    for result in results:
        print(result)

    elapsed = time.time() - start
    print(f"Time: {elapsed:.2f}s (all greetings run together)")

await greet_multiple()

Sequential Execution
Hello Alice
Hello Bob
Hello Charlie
Time: 3.01s (each greeting waits for previous)

Concurrent Execution
Hello Alice
Hello Bob
Hello Charlie
Time: 1.00s (all greetings run together)


### Common Mistake: Forgetting `await`

This creates a silent bug that's hard to catch!

In [33]:
# Common mistake: forgetting await
async def get_data():
    await asyncio.sleep(0.5)
    return "important data"

async def demonstrate_forgot_await():
    """Show what happens when you forget await"""

    # Wrong - forgot await
    print("WRONG: Forgot await")
    result = get_data() # Returns coroutine object, doesn't execute
    print(f"Type: {type(result)}")
    print(f"Value: {result}")
    print("This is NOT the data we wanted")

    # Correct - Using await
    print("\nCORRECT: Using await")
    result = await get_data()
    print(f"Type: {type(result)}")
    print(f"Value: {result}")
    print("This IS the data we wanted")

await demonstrate_forgot_await()

WRONG: Forgot await
Type: <class 'coroutine'>
Value: <coroutine object get_data at 0x110df34c0>
This is NOT the data we wanted

CORRECT: Using await
Type: <class 'str'>
Value: important data
This IS the data we wanted


/var/folders/mt/qblgswcj4rs5ll70_77j9_nw0000gn/T/ipykernel_80486/4268021733.py:18: RuntimeWarning: coroutine 'get_data' was never awaited
  result = await get_data()


## 6. Running Multiple Tasks Concurrently

### Three main approaches:

1. `asyncio.gather()` - Run all tasks, collect all results
2. `asyncio.create_task()` - Start task but don't wait immediately
3. `asyncio.wait()` - Advanced control (wait for first/all)

### 1. asyncio.gather() - Run all tasks concurrently

In [35]:
# asyncio.gather() - Run all tasks concurrently
async def fetch_user(user_id):
    """Simulate fetching user data from API"""
    await asyncio.sleep(1)  # Simulate network delay
    return {
        "id" : user_id,
        "name" : f"User {user_id}"
    }

async def gather_example():
    """Demonstrate asyncio.gather()"""
    print("asyncio.gather()")
    start = time.time()

    # Fetch 5 users concurrently
    users = await asyncio.gather(
        fetch_user(1),
        fetch_user(2),
        fetch_user(3),
        fetch_user(4),
        fetch_user(5),
    )

    elapsed = time.time() - start

    print(f"Fetched {len(users)} users:")
    for user in users:
        print(f"    - {user}")

    print(f"\nTime: {elapsed:.2f}s (all fetched concurrently)")
    print(f"If sequential: ~5s (1s x 5)")

await gather_example()


asyncio.gather()
Fetched 5 users:
    - {'id': 1, 'name': 'User 1'}
    - {'id': 2, 'name': 'User 2'}
    - {'id': 3, 'name': 'User 3'}
    - {'id': 4, 'name': 'User 4'}
    - {'id': 5, 'name': 'User 5'}

Time: 1.00s (all fetched concurrently)
If sequential: ~5s (1s x 5)


### 2. asyncio.create_task() - Fire and forget, then await later

In [36]:
# asyncio.create_task() - Fire and forget, then await later
async def background_task(name, duration):
    """Simulate background work"""
    print(f"{name}: starting background work")
    await asyncio.sleep(duration)
    print(f"{name}: Background work done")
    return f"{name} completed"

async def create_task_example():
    """Demonstrate asyncio.create_task()"""
    print("asyncio.create_task()")

    # Start tasks but don't wait yet
    task1 = asyncio.create_task(background_task("Task 1", 2))
    task2 = asyncio.create_task(background_task("Task 2", 1))

    # Do other work while tasks run in background
    print("Main: Doing other work")
    await asyncio.sleep(0.5)
    print("Main: Still working")
    await asyncio.sleep(0.5)
    print("Main: Almost done")

    # Now wait for background tasks to complete
    result1 = await task1
    result2 = await task2

    print(f"\nResults: {result1}, {result2}")

await create_task_example()

asyncio.create_task()
Main: Doing other work
Task 1: starting background work
Task 2: starting background work
Main: Still working
Task 2: Background work done
Main: Almost done
Task 1: Background work done

Results: Task 1 completed, Task 2 completed


## 7. Error Handling in Async Code

Errors in async code work like synchronous code, with some nuances.

### Error handling for single task

In [37]:
# Error handling with try/except
async def risky_task(task_id, should_fail):
    """Task that might fail"""
    await asyncio.sleep(0.5)
    if should_fail:
        raise ValueError(f"Task {task_id} failed")
    return f"Task {task_id} succeeded"

async def error_handling_basic():
    """Basic error handling in async functions"""
    print("Basic Error Handling")

    # Single task with error handling
    try:
        result = await risky_task(1, should_fail = True)
        print(result)
    except ValueError as e:
        print(f"Caught error: {e}")

await error_handling_basic()

Basic Error Handling
Caught error: Task 1 failed


### Error handling for multiple tasks

In [39]:
# Error handling with gather()
async def error_handling_gather():
    """Error handling with multiple tasks"""

    # Default behavior - stops on first error
    print("gather() - Default (stops on error)")
    try:
        results = await asyncio.gather(
            risky_task(1, should_fail = False),
            risky_task(2, should_fail = True),
            risky_task(3, should_fail = False)
        )
    except ValueError as e:
        print(f"Error: {e}")
        print("Task 1 and 3 results are lost")

    # return_exception=True - collect all results including errors
    print("\ngather() - return_exceptions=True")
    results = await asyncio.gather(
        risky_task(1, should_fail = False),
        risky_task(2, should_fail = True),
        risky_task(3, should_fail = False),
        return_exceptions = True
    )

    for i, result in enumerate(results, 1):
        if isinstance(result, Exception):
            print(f"Task {i}: Failed - {result}")
        else:
            print(f"Task {i}: {result}")

await error_handling_gather()

gather() - Default (stops on error)
Error: Task 2 failed
Task 1 and 3 results are lost

gather() - return_exceptions=True
Task 1: Task 1 succeeded
Task 2: Failed - Task 2 failed
Task 3: Task 3 succeeded


## 8. Timeouts

Always set timeouts for external operations to prevent hanging indefinitely.

In [40]:
# Using asyncio.wait_for() for timeouts
async def slow_operation():
    """Operation that takes 3 seconds"""
    print("Starting slow operation")
    await asyncio.sleep(3)

async def timeout_example():
    """Demonstrate timeout handling"""
    
    # This will timeout
    try:
        result = await asyncio.wait_for(
            slow_operation(),
            timeout = 2.0   # Wait maximum 2 seconds
        )
        print(result)
    except asyncio.TimeoutError:
        print("Operation timed out after 2 seconds")

await timeout_example()

Starting slow operation
Operation timed out after 2 seconds


> Timeouts prevent operations from hanging forever

## 9. Async Context Managers

Use `async with` for resources that need async setup/cleanup.

In [43]:
# Creating async context manager
class AsyncDatabase:
    """Simulate async database connection"""

    async def __aenter__(self):
        print("Opening database connection")
        await asyncio.sleep(0.5)    # Simulate connection time
        print("Database connected")
        self.connection = "DB Connection"
        return self

    async def __aexit__(self, exc_type, exc_val, exc_tb):
        print("Closing database connection")
        await asyncio.sleep(0.3)    # Simulate cleanup time
        print("Database disconnected")

    async def query(self, sql):
        """Simulate database query"""
        await asyncio.sleep(0.2)
        return f"Results for : {sql}"

async def async_context_manager_example():
    """Demonstrate async context manager"""
    print("Async Context Manager")

    async with AsyncDatabase() as db:
        result1 = await db.query("SELECT * FROM users")
        print(f"    {result1}")

        result2 = await db.query("SELECT * FROM posts")
        print(f"    {result2}")

    print("\nDatabase automaticall closed even if error occurred")

await async_context_manager_example()

Async Context Manager
Opening database connection
Database connected
    Results for : SELECT * FROM users
    Results for : SELECT * FROM posts
Closing database connection
Database disconnected

Database automaticall closed even if error occurred


## 10. Async Iterators

Use `async for` to iterate over data that arrives asynchronously.

In [45]:
# Async generator
async def async_range(n):
    """Async generator that yields number with delay"""
    for i in range(n):
        await asyncio.sleep(0.3)    # Simulate async operation
        yield i

async def async_iterator_example():
    """Demonstrate async iteration"""
    print("Async Iterator")

    async for num in async_range(5):
        print(f"Received: {num}")

    print("\nEach number arrived after async operation")

await async_iterator_example()

Async Iterator
Received: 0
Received: 1
Received: 2
Received: 3
Received: 4

Each number arrived after async operation


## 11. Common Pitfalls

### Pitfall 1: Blocking the Event Loop

The BIGGEST mistake in async programming!

In [46]:
# DON'T DO THIS - Blocking the event loop
async def blocking_bad():
    """BAD: Using time.sleep() in async function"""
    print("Task 1: Start")
    time.sleep(2)   # BLOCK THIS ENTIRE EVENT LOOP
    print("Task 1: Done")

async def blocking_good():
    """GOOD: Using asyncio.sleep()"""
    print("Task 2: Start")
    await asyncio.sleep(2)  # Non-blocking, allows other tasks
    print("Task 2: Done")

async def demonstrate_blocking():
    """Show the difference between blocking and non-blocking"""

    # This will be slow (Sequential)
    print("BAD: Using time.sleep()")
    start = time.time()

    # Even though we use gather, time.sleep() blocks
    await asyncio.gather(
        blocking_bad(),
        blocking_bad()
    )

    print(f"Time: {time.time() - start:.2f}s (blocking)\n")

    # This will be fast (concurrent)
    print("GOOD: Using asyncio.sleep()")
    start = time.time()

    await asyncio.gather(
        blocking_good(),
        blocking_good()
    )

    print(f"Time: {time.time() - start:.2f}s (concurrent)")

await demonstrate_blocking()

BAD: Using time.sleep()
Task 1: Start
Task 1: Done
Task 1: Start
Task 1: Done
Time: 4.01s (blocking)

GOOD: Using asyncio.sleep()
Task 2: Start
Task 2: Start
Task 2: Done
Task 2: Done
Time: 2.00s (concurrent)


### Pitfall 2: Creating Too Many Tasks

Limit concurrent tasks with semaphores.

#### simpler way - code

In [47]:
# Using sempaphore to limit concurrency
async def limited_concurrency_example():
    """Demonstrate limiting concurrent tasks"""

    # Semaphore limits concurrent tasks
    semaphore = asyncio.Semaphore(3)    # Max 3 concurrent

    async def limited_task(task_id):
        async with semaphore:
            print(f"Task {task_id}: Running (max 3 at once)")
            await asyncio.sleep(1)
            print(f"Task {task_id}: Done")
            return task_id

    print("Semaphore (max 3 concurrent)")

    tasks = []

    # create 10 tasks explicitly
    for i in range(10):
        task = limited_task(i)
        tasks.append(task)

    # Run tasks concurrently (but limited by semaphore)
    results = await asyncio.gather(*tasks)

    print(f"\nCompleted {len(results)} tasks (3 at a time)")

await limited_concurrency_example()

Semaphore (max 3 concurrent)
Task 0: Running (max 3 at once)
Task 1: Running (max 3 at once)
Task 2: Running (max 3 at once)
Task 0: Done
Task 1: Done
Task 2: Done
Task 3: Running (max 3 at once)
Task 4: Running (max 3 at once)
Task 5: Running (max 3 at once)
Task 3: Done
Task 4: Done
Task 5: Done
Task 6: Running (max 3 at once)
Task 7: Running (max 3 at once)
Task 8: Running (max 3 at once)
Task 6: Done
Task 7: Done
Task 8: Done
Task 9: Running (max 3 at once)
Task 9: Done

Completed 10 tasks (3 at a time)


#### python way - code

In [50]:
# Using sempaphore to limit concurrency
async def limited_concurrency_example():
    """Demonstrate limiting concurrent tasks"""

    # Semaphore limits concurrent tasks
    semaphore = asyncio.Semaphore(3)    # Max 3 concurrent

    async def limited_task(task_id):
        async with semaphore:
            print(f"Task {task_id}: Running (max 3 at once)")
            await asyncio.sleep(1)
            print(f"Task {task_id}: Done")
            return task_id

    print("Semaphore (max 3 concurrent)")

    # Create 10 tasks, but only 3 run concurrently
    tasks = [limited_task(i) for i in range(10)]
    results = await asyncio.gather(*tasks)

    print(f"\nCompleted {len(results)} tasks (3 at a time)")

await limited_concurrency_example()

Semaphore (max 3 concurrent)
Task 0: Running (max 3 at once)
Task 1: Running (max 3 at once)
Task 2: Running (max 3 at once)
Task 0: Done
Task 1: Done
Task 2: Done
Task 3: Running (max 3 at once)
Task 4: Running (max 3 at once)
Task 5: Running (max 3 at once)
Task 3: Done
Task 4: Done
Task 5: Done
Task 6: Running (max 3 at once)
Task 7: Running (max 3 at once)
Task 8: Running (max 3 at once)
Task 6: Done
Task 7: Done
Task 8: Done
Task 9: Running (max 3 at once)
Task 9: Done

Completed 10 tasks (3 at a time)


## 12. Real-World Example: Fetching Multiple URLs

Let's combine everything we learned into a practical example.

In [52]:
# Real-world example: Concurrent HTTP requests
async def fetch_url_data(session_id, url_id):
    """Simulate fetching data from URL"""
    delay = 0.5 + (url_id % 3) * 0.3    # Variable delays

    print(f"[Session {session_id}] Fetching URL {url_id}")
    await asyncio.sleep(delay)

    return {
        "session" : session_id,
        "url_id": url_id,
        "data": f"Data from URL {url_id}",
        "delay": delay 
    }

async def real_world_example():
    """Fetch data from multiple URLs concurrently"""
    print("Real-World: Concurrently URL Fetching")

    start = time.time()

    # Fetching 10 URLs with max 4 concurrent requests
    semaphore = asyncio.Semaphore(4)

    async def fetch_with_limit(url_id):
        async with semaphore:
            return await fetch_url_data(1, url_id)

    # Create tasks for all URLs
    tasks = [fetch_with_limit(i) for i in range(10)]

    # Fetch all with error handling
    results = await asyncio.gather(*tasks, return_exceptions=True)

    elapsed = time.time() - start

    # Process results
    print(f"\nResults")
    successful = 0
    failed = 0

    for result in results:
        if isinstance(result, Exception):
            failed += 1
        else:
            successful += 1

    print(f"Successful: {successful}")
    print(f"Failed: {failed}")
    print(f"Total time: {elapsed:.2f}s")
    print(f"If sequential: ~{sum(r.get('delay', 0) for r in results if isinstance(r, dict)):.2f}s")

await real_world_example()

Real-World: Concurrently URL Fetching
[Session 1] Fetching URL 0
[Session 1] Fetching URL 1
[Session 1] Fetching URL 2
[Session 1] Fetching URL 3
[Session 1] Fetching URL 4
[Session 1] Fetching URL 5
[Session 1] Fetching URL 6
[Session 1] Fetching URL 7
[Session 1] Fetching URL 8
[Session 1] Fetching URL 9

Results
Successful: 10
Failed: 0
Total time: 2.40s
If sequential: ~7.70s


## Performance Comparison Summary

Let's create a final comparison of sync vs async performance.

In [55]:
# FInal performance comparison
async def performance_summary():
    """Summary of sync vs async performance"""

    # Simulate 10 API calls
    num_calls = 10
    delay_per_call = 1.0

    # Synchronous version
    print("\n1. SYNCHRONOUS (blocking)")
    start = time.time()
    for i in range(num_calls):
        time.sleep(delay_per_call)
    sync_time = time.time() - start
    print(f"    Time: {sync_time:.2f}s")
    print(f"    = {num_calls} calls × {delay_per_call}s each")

    # Asynchronous version
    print("\n2. ASYNCHRONOUS (concurrent)")
    start = time.time()
    
    tasks = []

    # Create async sleep tasks
    for i in range(num_calls):
        task = asyncio.sleep(delay_per_call)
        tasks.append(task)

    # Run all tasks concurrently
    await asyncio.gather(*tasks)

    async_time = time.time() - start

    print(f"    Time: {async_time:.2f}s")
    print(f"    = All {num_calls} calls run together")

    # Comparison
    print("\n3. COMPARISON")
    speedup = sync_time / async_time
    print(f"    Speedup: {speedup:.1f}x faster")
    print(f"   Time saved: {sync_time - async_time:.2f}s")

await performance_summary()


1. SYNCHRONOUS (blocking)
    Time: 10.03s
    = 10 calls × 1.0s each

2. ASYNCHRONOUS (concurrent)
    Time: 1.00s
    = All 10 calls run together

3. COMPARISON
    Speedup: 10.0x faster
   Time saved: 9.03s


## Key Takeaways

1. **Async enables concurrency, not parallelism** - Tasks make progress together on one CPU

2. **The event loop manages task scheduling** - Switches between tasks when they're waiting

3. **Use async for I/O-bound tasks** - Network, database, files (not CPU-intensive work)

4. **Always await your coroutines** - Forgetting `await` creates silent bugs

5. **Use `asyncio.gather()` for concurrent operations** - Run multiple tasks at once

6. **Set timeouts on external operations** - Prevent hanging indefinitely

7. **Don't block the event loop** - Use `asyncio.sleep()` not `time.sleep()`

8. **Limit concurrency with semaphores** - Prevent overwhelming resources

9. **Handle errors properly** - Use `return_exceptions=True` to collect all results

10. **Async shines for I/O, not CPU work** - Choose the right tool for the job

## Next Steps

- Practice with the exercises in the Day 5 notes
- Try building a small async web scraper
- Read about async context managers and iterators
- Explore `aiohttp` for real HTTP requests

## Additional Resources

- [Python asyncio Documentation](https://docs.python.org/3/library/asyncio.html)
- [Real Python - Async IO](https://realpython.com/async-io-python/)
- [aiohttp Documentation](https://docs.aiohttp.org/)
- Day 5 detailed notes for more examples and patterns